# 02 — Prepare Datasets (`uci` pilot group)

**Purpose.** Download, split, and scale the 12 datasets in the `uci` group,
using the **real** upstream download/parsing logic (imported directly from
the cloned repo, not re-derived) combined with a **faithful, unit-tested
port** of the exact split/scale arithmetic from `BaseDataModule.load_datasets`
-- including the easy-to-miss "cap calibration at 2048 samples, redistribute
the excess" rule.

**Expected runtime:** a few minutes the first time (network-bound downloads
of ~12 small files/archives), near-instant on reruns (cached to disk).
**GPU:** not used in this notebook.

**A note on how this was verified:** the network domain UCI's archive lives
on is not reachable from the sandbox this project was authored in, so the
split/scale logic below was verified with synthetic data standing in for
real downloads (16 unit tests, all passing — see `tests/test_uci_pilot.py`),
and the real download wrapper was separately verified to correctly reach
the actual upstream network call before failing on the sandbox's network
restriction (confirming the code path itself, not just its logic, is
correct). It has **not** been verified against a live UCI download from
this authoring environment — that will happen for real the first time you
run this cell on Colab, which has open internet access.

**A real compatibility bug already fixed for you:** both upstream repos pin
`openml==0.13.1` exactly. Actually running that version here fails under
NumPy 2.0 (`np.sctypes` was removed). `requirements.txt` now installs a
modern `openml` instead — confirmed working (reaches the real network layer
correctly) in this sandbox.


## Step 1 — Locate project, import helpers, clone external repo if missing

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("PROJECT_ROOT not found. Run 00_environment.ipynb first.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import env_utils
from src.datasets import uci_pilot

EXTERNAL_DIR = PROJECT_ROOT / "external" / "quantile-recalibration-training"
if not EXTERNAL_DIR.exists():
    print("External repo not found -- cloning now (normally done by notebook 01).")
    env_utils.clone_or_pull_repo(
        repo_url="https://github.com/Vekteur/quantile-recalibration-training.git",
        dest=EXTERNAL_DIR, branch="main",
    )
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"External repo present: {EXTERNAL_DIR.exists()}")


## Step 2 — Load config (split ratio, seed, pilot dataset list)

In [ ]:
import yaml

cfg = yaml.safe_load((PROJECT_ROOT / "configs" / "config.yaml").read_text())
SPLIT_RATIO = tuple(cfg["data"]["train_inter_val_calib_test_split_ratio"])
PILOT_DATASETS = tuple(cfg["data"]["uci_group_datasets"])
SEED = 0  # matches upstream's default seed=0

assert SPLIT_RATIO == uci_pilot.DEFAULT_SPLIT_RATIO, "config.yaml split ratio drifted from the verified default"
assert PILOT_DATASETS == uci_pilot.UCI_PILOT_DATASETS, "config.yaml uci group drifted from the verified list"

print(f"Split ratio (train/inter/val/calib/test): {SPLIT_RATIO}")
print(f"Pilot datasets ({len(PILOT_DATASETS)}): {PILOT_DATASETS}")
print(f"Seed: {SEED}")


## Step 3 — Download, split, and scale all 12 pilot datasets

This is the one cell in this notebook that needs real internet access to
`archive.ics.uci.edu` (and OpenML, for `Kin8nm` specifically) — it will not
succeed in a network-sandboxed environment. On Colab it should just work.

Results are cached to `data/uci/<name>/{x,y}.npy` by the upstream download
function itself, so re-running this cell after the first successful run is
fast (no re-download).

In [ ]:
DATA_DIR = PATHS["outputs"].parent / "data" if "PATHS" in dir() else PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

prepared = uci_pilot.prepare_uci_pilot_group(
    project_root=PROJECT_ROOT,
    data_dir=DATA_DIR,
    dataset_names=PILOT_DATASETS,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
)
print(f"\nPrepared {len(prepared)}/{len(PILOT_DATASETS)} datasets.")


## Step 4 — Sanity-check the prepared splits

Cross-checks against Table 3 of the paper (training-instance counts) as an
independent correctness signal, not just "did it run without an exception".

In [ ]:
import pandas as pd

# From Table 3 of the paper (Appendix L), 'Nb of training instances' column,
# for exactly the 12 `uci` group datasets -- typed in once, checked here.
paper_table3_train_counts = {
    "CPU": 135, "Yacht": 200, "MPG": 254, "Energy": 499, "Crime": 531, "Fish": 590,
    "Concrete": 669, "Airfoil": 976, "Kin8nm": 5324, "Power": 6219, "Naval": 7757, "Protein": 31328,
}

rows = []
mismatches = []
for name, splits in prepared.items():
    n_train = len(splits["train"]["x"])
    n_total = sum(len(splits[s]["x"]) for s in splits)
    expected = paper_table3_train_counts[name]
    match = (n_train == expected)
    if not match:
        mismatches.append(name)
    rows.append({
        "dataset": name, "train": n_train, "val": len(splits["val"]["x"]),
        "calib": len(splits["calib"]["x"]), "test": len(splits["test"]["x"]),
        "total": n_total, "paper_table3_train": expected, "matches_paper": match,
    })

df = pd.DataFrame(rows).set_index("dataset")
print(df)

if mismatches:
    print(f"\n[warn] Train-count mismatch vs paper Table 3 for: {mismatches}. "
          "Could be a real discrepancy (e.g. a UCI source file changed since 2024) "
          "or a preprocessing difference -- worth a manual look before trusting PCE "
          "comparisons on these specific datasets.")
else:
    print("\nAll 12 train-split sizes match paper Table 3 exactly.")


## Step 5 — Save prepared splits for notebook 03

In [ ]:
import pickle

out_path = (PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs") / "uci_pilot_splits.pkl"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "wb") as f:
    pickle.dump({"splits": prepared, "split_ratio": SPLIT_RATIO, "seed": SEED}, f)
print(f"Saved to {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")


## Next steps

**Next:** `03_train_pilot_models.ipynb` — trains the small pilot models
(one per dataset, `mixture_size=1`: single-Gaussian output, matching the
AISTATS 2024 paper's Appendix A ablation branch, which is the branch whose
assumptions actually match Neural Regression Collapse theory). Confirmed in
notebook 01: no pretrained weights exist to load, so this notebook trains,
using the exact architecture in `uq/models/general/mlp.py` /
`uq/models/pred_type/mixture_dist.py` read directly from the repo.
